In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [5]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # mean and std dev for CIFAR10 dataset to normalize to (-1,1)
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:56<00:00, 3.03MB/s] 


In [6]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [21]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # input channels = 3(RGB), Output channels = num. of feature maps(32) => 32 filters were applied
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2 

            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2    
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2) # kernel size = 2, stride = 2       
        ) 
        
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256), # flatten array size = 4*4*128 (explained in debashish's notes)
            nn.ReLU(),
            
            nn.Linear(256, 10), # output=10, for 10 classes
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flatten x
        x = self.fc_layers(x)
            
        return x

In [22]:
model = CNN()

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Train the CNN

In [26]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # Forward propagation
        loss = criterion(output, labels) # Loss function
        loss.backward() # Backward propagation
        optimizer.step() # Update params
        
        epoch_training_loss += loss.item()
        
    print(f"epoch{epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch1/10 & loss=0.6625130471518582
epoch2/10 & loss=0.5492584431720207
epoch3/10 & loss=0.4591287108676513
epoch4/10 & loss=0.3699187064052695
epoch5/10 & loss=0.2870403908936264
epoch6/10 & loss=0.23013442068282144
epoch7/10 & loss=0.1740649391461135
epoch8/10 & loss=0.14640977128368357
epoch9/10 & loss=0.12162718657955594
epoch10/10 & loss=0.10990430398717942


### Evaluate

In [ ]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1) # softmax  => maximum predicted values
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
        
print(f"accuracy = {correct_labels/total_labels * 100}")

accuracy = 74.85000000000001
